In [ ]:
def make_road_df(log_filepath):
    """
    Puts candump data into a dataframe with columns 'time', 'aid', and 'data'
    """
    road_df = pd.read_fwf(
        log_filepath, delimiter = ' '+ '#' + '('+')',
        skiprows = 0,skipfooter=0,
        usecols = [0,2,3],
        dtype = {0:'float64', 1:str, 2: str},
        names = ['time','aid', 'data'] )
    road_df.aid = road_df.aid.apply(lambda x: int(x,16))
    road_df.data = road_df.data.apply(lambda x: x.zfill(16)) #pad with 0s on the left for data with dlc < 8
    road_df.time = road_df.time - road_df.time.min()
    return road_df[road_df.aid<=0x700]

In [4]:
import pandas as pd
dir = "road/attacks"
csa1_df = make_road_df(f"{dir}/correlated_signal_attack_1.log")
csa2_df = make_road_df(f"{dir}/correlated_signal_attack_2.log")
csa3_df = make_road_df(f"{dir}/correlated_signal_attack_3.log")
mecta_df = make_road_df(f"{dir}/max_engine_coolant_temp_attack.log") 
fa1_df= make_road_df(f"{dir}/fuzzing_attack_1.log")
fa2_df= make_road_df(f"{dir}/fuzzing_attack_2.log")
fa3_df= make_road_df(f"{dir}/fuzzing_attack_3.log")
msa1_df = make_road_df(f"{dir}/max_speedometer_attack_1.log")
msa2_df = make_road_df(f"{dir}/max_speedometer_attack_2.log")
msa3_df = make_road_df(f"{dir}/max_speedometer_attack_3.log")
rloffa1_df = make_road_df(f"{dir}/reverse_light_off_attack_1.log")
rloffa2_df = make_road_df(f"{dir}/reverse_light_off_attack_2.log")
rloffa3_df = make_road_df(f"{dir}/reverse_light_off_attack_3.log")
rlona1_df = make_road_df(f"{dir}/reverse_light_on_attack_1.log")
rlona2_df = make_road_df(f"{dir}/reverse_light_on_attack_2.log")
rlona3_df = make_road_df(f"{dir}/reverse_light_on_attack_3.log")

In [ ]:
import re
import pandas as pd
def label_and_format_dataframe(df: pd.DataFrame, injection_data_str: str, output_csv: str = None):
    """
    Formats a parsed CAN DataFrame and labels messages based on a wildcard-aware injection pattern.

    Args:
        df (pd.DataFrame): Input DataFrame with columns ['time', 'aid', 'data']
        injection_data_str (str): Injection string with possible 'X' wildcards (e.g., '59XX45XX0000FFFF')
        output_csv (str, optional): Path to save the labeled output CSV (if given)

    Returns:
        pd.DataFrame: Labeled and formatted DataFrame
    """

    # Copy input DataFrame and standardize
    df = df.copy()
    df['data'] = df['data'].str.upper().str.zfill(16)

    # Split into DATA[0] to DATA[7]
    for i in range(8):
        df[f'DATA[{i}]'] = df['data'].str[i*2:i*2+2].apply(lambda x: int(x, 16))

    # Add DLC = 8
    df['DLC'] = 8

    # Build regex pattern with wildcard support
    inj = injection_data_str.upper().zfill(16)
    pattern = ''.join(['..' if 'X' in inj[i:i+2] else inj[i:i+2] for i in range(0, len(inj), 2)])
    regex = re.compile(f"^{pattern}$")

    # Assign 1 for malicious (matches), 0 for benign
    df['Flag'] = df['data'].apply(lambda x: 1 if regex.match(x) else 0)

    # Rename columns
    df.rename(columns={'time': 'Timestamp', 'aid': 'CAN ID'}, inplace=True)

    # Reorder columns
    ordered_cols = ['Timestamp', 'CAN ID', 'DLC'] + [f'DATA[{i}]' for i in range(8)] + ['Flag']
    df = df[ordered_cols]
    df.drop('Timestamp', axis=1, inplace=True)
    # display(df['Flag'].value_counts())
    if output_csv:
        df.to_csv(output_csv, index=False)
        # print(f" Saved labeled file to: {output_csv}")
    return df


In [7]:
df_injections = {
    "csa1":      (csa1_df, "595945450000FFFF"),
    "csa2":      (csa2_df, "595945450000FFFF"),
    "csa3":      (csa3_df, "595945450000FFFF"),
    "fa1":       (fa1_df, "FFFFFFFFFFFFFFFF"),
    "fa2":       (fa2_df, "FFFFFFFFFFFFFFFF"),
    "fa3":       (fa3_df, "FFFFFFFFFFFFFFFF"),
    "msa1":      (msa1_df, "XXXXXXXXXXFFXXXX"),
    "msa2":      (msa2_df, "XXXXXXXXXXFFXXXX"),
    "msa3":      (msa3_df, "XXXXXXXXXXFFXXXX"),
    "mecta":     (mecta_df, "XXXXXXXXXXFFXXXX"),
    "rloffa1":   (rloffa1_df, "XXXX04XXXXXXXXXX"),
    "rloffa2":   (rloffa1_df, "XXXX04XXXXXXXXXX"),
    "rloffa3":   (rloffa1_df, "XXXX04XXXXXXXXXX"),
    "rlona1":    (rlona2_df, "XXXX0CXXXXXXXXXX"),
    "rlona2":    (rlona2_df, "XXXX0CXXXXXXXXXX"),
    "rlona3":    (rlona2_df, "XXXX0CXXXXXXXXXX"),
}

# Loop through and apply the function
for name, (df, inj_str) in df_injections.items():
    output_path = f"preprocessed/{name}.csv"
    label_and_format_dataframe(df, inj_str, output_csv=output_path)

In [ ]:
import glob
import os
import pandas as pd

csv_folder = 'preprocessed'
csv_files = glob.glob(os.path.join(csv_folder, "*.csv"))
attack_df = pd.concat((pd.read_csv(f) for f in csv_files), ignore_index=True)
attack_df.to_csv("results/attack_data.csv", index=False)

print(f"Merged {len(csv_files)} files into attack_data.csv")
# To check the files
csv_files = [f for f in csv_files if not os.path.basename(f).rstrip(".csv").endswith("m")]
print(csv_files)
display(attack_df['Flag'].value_counts())

Merged 16 files into attack_data.csv
['preprocessed/csa1.csv', 'preprocessed/fa3.csv', 'preprocessed/fa2.csv', 'preprocessed/csa2.csv', 'preprocessed/fa1.csv', 'preprocessed/csa3.csv', 'preprocessed/msa2.csv', 'preprocessed/msa3.csv', 'preprocessed/msa1.csv', 'preprocessed/rloffa3.csv', 'preprocessed/rloffa2.csv', 'preprocessed/rloffa1.csv', 'preprocessed/rlona3.csv', 'preprocessed/rlona2.csv', 'preprocessed/rlona1.csv', 'preprocessed/mecta.csv']


Flag
0    1501845
1      49663
Name: count, dtype: int64

1551508
